In [83]:
import pandas as pd
import json
from pathlib import Path

In [84]:
# Path("data/raw") specifies the directory containing your JSON files
# .rglob("*.json") recursively searches this directory AND its subdirectories
# for all files ending in .json
# sorted() puts the file paths in a consistent order
# files becomes a list of Path objects pointing to your JSON files as sorted by default returns a list

files = sorted(Path("data/raw").rglob("*.json"))

In [85]:
print(f"Found {len(files)} JSON files")

Found 10 JSON files


In [86]:
# files[0] selects the FIRST JSON file in the list
# open(..., "r") opens the file in read mode
# "as f" assigns the opened file object to the variable f
# The 'with' statement automatically closes the file when finished

with open(files[0], "r") as f:
    data = json.load(f)

In [87]:
data.keys()

dict_keys(['search_metadata', 'search_parameters', 'best_flights', 'other_flights', 'price_insights', 'airports'])

In [88]:
# defining flights as the best_flights and other_flights nested data in json, then normalizing it and setting it as our df. [] means that if best_flights or other_flights doesnt exist then it returns an empty list rather than error
flights = (
    data.get("best_flights", []) +
    data.get("other_flights", [])
)

df = pd.json_normalize(flights)

In [89]:
df.head()

,flights,layovers,total_duration,price,type,airline_logo,departure_token,carbon_emissions.this_flight,carbon_emissions.typical_for_this_route,carbon_emissions.difference_percent
0,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 425, 'name': 'Dubai Internationa...",1255,1618,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1429000,1242000,15
1,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 160, 'name': 'Dubai Internationa...",980,1622,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1475000,1242000,19
2,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 95, 'name': 'Dubai International...",925,1692,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1511000,1242000,22
3,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 130, 'name': 'Istanbul Airport',...",950,1862,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1303000,1242000,5
4,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 560, 'name': 'Zurich Airport', '...",1915,1530,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1365000,1242000,10


In [90]:
# {} means it will return empty dictiornay if no search_parameters, not an error
params = data.get("search_parameters", {})

In [91]:
df["departure_airport"] = params.get("departure_id")
df["arrival_airport"] = params.get("arrival_id")
df["outbound_date"] = params.get("outbound_date")
df["return_date"] = params.get("return_date")
df["currency"] = params.get("currency")

In [92]:
df.head()

,flights,layovers,total_duration,price,type,airline_logo,departure_token,carbon_emissions.this_flight,carbon_emissions.typical_for_this_route,carbon_emissions.difference_percent,departure_airport,arrival_airport,outbound_date,return_date,currency
0,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 425, 'name': 'Dubai Internationa...",1255,1618,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1429000,1242000,15,BHX,MRU,2027-01-07,2027-01-16,GBP
1,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 160, 'name': 'Dubai Internationa...",980,1622,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1475000,1242000,19,BHX,MRU,2027-01-07,2027-01-16,GBP
2,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 95, 'name': 'Dubai International...",925,1692,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1511000,1242000,22,BHX,MRU,2027-01-07,2027-01-16,GBP
3,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 130, 'name': 'Istanbul Airport',...",950,1862,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1303000,1242000,5,BHX,MRU,2027-01-07,2027-01-16,GBP
4,[{'departure_airport': {'name': 'Birmingham Ai...,"[{'duration': 560, 'name': 'Zurich Airport', '...",1915,1530,Round trip,https://www.gstatic.com/flights/airline_logos/...,WyJDalJJV0VwVlRVRlhjMVZuV0ZWQlpYQTRVSGRDUnkwdE...,1365000,1242000,10,BHX,MRU,2027-01-07,2027-01-16,GBP


In [93]:
df = df[[
    "flights",
    "departure_airport",
    "arrival_airport",
    "outbound_date",
    "return_date",
    "total_duration",
    "price",
    "currency",
    "type"
]]

In [94]:
df.head()

,flights,departure_airport,arrival_airport,outbound_date,return_date,total_duration,price,currency,type
0,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1255,1618,GBP,Round trip
1,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,980,1622,GBP,Round trip
2,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,925,1692,GBP,Round trip
3,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,950,1862,GBP,Round trip
4,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1915,1530,GBP,Round trip


In [96]:
from datetime import datetime, timezone

In [98]:
# get todays date
extraction_date = datetime.now(timezone.utc).date()

# Create extraction date column in dataframe
df["extraction_date"] = extraction_date

In [99]:
df.head()

,flights,departure_airport,arrival_airport,outbound_date,return_date,total_duration,price,currency,type,extraction_date
0,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1255,1618,GBP,Round trip,2026-10-09
1,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,980,1622,GBP,Round trip,2026-10-09
2,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,925,1692,GBP,Round trip,2026-10-09
3,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,950,1862,GBP,Round trip,2026-10-09
4,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1915,1530,GBP,Round trip,2026-10-09


In [100]:
import hashlib

In [101]:
# Function to generate a deterministic hash for each row
def generate_id(row):

    # Select the fields that identify this observation
    record = {
        "departure_airport": row["departure_airport"],
        "arrival_airport": row["arrival_airport"],
        "outbound_date": row["outbound_date"],
        "return_date": row["return_date"],
        "price": int(row["price"]),
        "currency": row["currency"],
        "extraction_date": str(row["extraction_date"]),
        "flights": row["flights"]
    }

    # Convert the dictionary into a consistent JSON string
    record_string = json.dumps(
        record,
        sort_keys=True,
        default=str
    )

    # Generate a SHA-256 hash and return it as a string
    return hashlib.sha256(
        record_string.encode("utf-8")
    ).hexdigest()

In [102]:
# Apply the function to every row in your DataFrame, axis=1 says we operate across rows instead of columns
df["observation_id"] = df.apply(generate_id, axis=1)

In [103]:
df.head()

,flights,departure_airport,arrival_airport,outbound_date,return_date,total_duration,price,currency,type,extraction_date,observation_id
0,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1255,1618,GBP,Round trip,2026-10-09,4251b0e17588dcf9bf5e32826925232ebed3b1e01ef072...
1,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,980,1622,GBP,Round trip,2026-10-09,23f459608ed7281d645adebaf64a0e8c354387ab0d11e0...
2,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,925,1692,GBP,Round trip,2026-10-09,f38e9f9ebde5b2b614bb5485cbe722470fd5fb11a2835d...
3,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,950,1862,GBP,Round trip,2026-10-09,aec32f5ccc7bb8b6e10809677563c6f72a75731cd9ec59...
4,[{'departure_airport': {'name': 'Birmingham Ai...,BHX,MRU,2027-01-07,2027-01-16,1915,1530,GBP,Round trip,2026-10-09,699c8fc76419d6a52a68ed3ed9b2ba8804f4f16e15b13d...
